# NLP Capstone Project: Retrieval-Based Chatbot

**By Faatimah Majothi**

## Introduction

In this project, we will build, train and test a simple retrieval-based chatbot called **YapBot**, as part of the capstone project for the [Data Scientist: Natural Language Processing Specialist Career Path]('https://www.codecademy.com/learn/paths/data-science-nlp') on Codecademy. We will be using a corpus of human conversation text sourced from [Kaggle](https://www.kaggle.com/datasets/projjal1/human-conversation-training-data/data) to train the chatbot to respond according to user input.

The project scope below outlines the workflow we aim to use to create our `YapBot`.

**Project Scope:**
- Introduction
- A: Setup
    - Import python libraries and data
    - Prepare and preprocess conversation data
- B: Build chatbot
    - Create `YapBot` chatbot class
    - Create and link functions within class
- C: Test and run completed `YapBot`
- Conclusion

## Section A: Setup

To begin with, we will **import the python libraries required** to process our text corpus. We will then **import, clean and prepare the text data** before moving on to building the chatbot.

##### **Initialise libraries:**

In [3]:
# Import libraries

%pip install streamlit spacy 
import streamlit as st
import string
import spacy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


##### **Import Data:**

In [4]:
# Load conversation data
with open('train-data/human_chat.txt', 'r') as file:
    lines = file.readlines()

conversations = []
current_convo = []

# Load spaCy model
word2vec = spacy.load('en_core_web_sm')

# Exit commands
exit_commands = ("quit", "goodbye", "exit", "no", "nothing", "never mind", "bye", "end", "stop")

##### **Create Setup Function:**

In [ ]:
# Create cleaning function
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return " ".join(text.split())

##### **Prepare and Preprocess Data:**

In [ ]:
# Preprocess data by conversation and question/answer
conversations = []
current_convo = []

for line in lines:
    if line.lower().startswith("human 1: Hi!") or line.lower().startswith("human 1: Hi"):
        if current_convo:
            conversations.append(current_convo)
            current_convo = []
    current_convo.append(line.strip())
if current_convo:
    conversations.append(current_convo)

questions = []
answers = []

for convo in conversations:
    for i in range(len(convo)-1):
        if convo[i].startswith("Human 1:") and convo[i+1].startswith("Human 2:"):
            q = convo[i].split(":", 1)[1].strip()
            a = convo[i+1].split(":", 1)[1].strip()
            questions.append(q)
            answers.append(a)

## B: Build chatbot

The text corpus is now cleaned and ready for training the chatbot. Before we can do so, we must **create the core functions** within the chatbot that will allow it to process input. Then we will **loop the functions** of our `YapBot` chatbot to simulate conversation.

##### **Build Chatbot:**

In [ ]:
class YapBot:
    def __init__(self, questions, answers, name="User"):
        self.questions = [clean_text(q) for q in questions]
        self.answers = answers
        self.name = name

    def compare_similarity(self, user_message, possible_response):
        user_doc = word2vec(user_message)
        response_doc = word2vec(possible_response)
        return user_doc.similarity(response_doc)

    def respond(self, user_message):
        cleaned_input = clean_text(user_message)
        similarity_scores = [self.compare_similarity(cleaned_input, q) for q in self.questions]
        if max(similarity_scores) < 0.7:
            return f"Sorry, {self.name}, I didn't understand that. Can you try again?"
        best_match_index = similarity_scores.index(max(similarity_scores))
        return self.answers[best_match_index]


## Section C: Testing 

##### **Test and Run Chatbot:**

Our chatbot is now complete. For the final step, we must test that the chatbot works as expected and resolve any errors. To do this, we will reproduce the above code chunks in a single python file ([Yapbot.py]('https://github.com/FaatimahMajothi98/NLP-Capstone-Chatbot/blob/master/NLP-Capstone/YapBot.py')) and add the following lines to connect the code to a streamlink interface:



> (#) Initialise YapBot

bot = YapBot(questions, answers)

> (#) Connect to Streamlit

st.title("YapBot")

if 'history' not in st.session_state:

    st.session_state['history'] = []

if 'initialized' not in st.session_state:

    st.session_state['history'].append(("YapBot", "Hello there! I'm YapBot. Ask me anything!"))
    st.session_state['initialized'] = True

user_input = st.text_input("Your questions here:", key="input")

if st.button("Send"):

    if user_input:

        response = bot.respond(user_input)
        st.session_state['history'].append(("You", user_input))
        st.session_state['history'].append(("YapBot", response))

for speaker, message in st.session_state['history']:

    st.write(f"**{speaker}:** {message}")


We can then run the streamlit app using this command in anly command line: `streamlink run YapBot.py`.

## Conclusion